# Boundary-driven Alfvén waves in a line-tied cavity

A straight guide field $B_0\hat z$ threads a finite interval $0\le z\le L$.
Small transverse displacement satisfies
$$\rho_0(z)\xi_{tt}=\frac{B_0^2}{\mu_0}\xi_{zz},\qquad
u_\perp=\xi_t,\quad b_\perp=B_0\xi_z.$$
We assume a static equilibrium with uniform pressure, constant guide field and
positive prescribed density. This is linear ideal shear-Alfvén dynamics, with
no compressive response or dissipation.

The **left footpoint is driven**, $\xi(0,t)=g(t)$; the **right is fixed**,
$\xi(L,t)=0$. Initially $\xi=\xi_t=0$. After the driver stops, both ends are
line-tied. No endpoint identification, periodic wrapping or additional magnetic
boundary condition is imposed. Density-gradient scattering and physical wall
reflections determine the evolution.

Use normalized units $L=B_0=\mu_0=\rho_L=1$, so time is measured in the
reference crossing time $L\sqrt{\mu_0\rho_L}/B_0$. Install
`python -m pip install -e '.[host,notebook,test]' -e './packages/models[precision,test]'` and select that kernel.


In [ ]:
import bspf_models.plasma.alfven as bspf_alfven
import pybspf.calculus as bspf_calculus
import pybspf.operators as bspf_operators
import pybspf.plans as bspf_plans

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import pybspf as b

amplitude, drive_duration = 1e-3, 0.5
times = jnp.linspace(0., 4., 401)
density = lambda z: 1 + 1.5*(1+jnp.tanh((z-0.5)/0.08))

def drive(t):
    return amplitude*jnp.where((t > 0) & (t < drive_duration),
        jnp.sin(jnp.pi*t/drive_duration)**8, 0.)

boundary = lambda t: jnp.array([drive(t), 0.])


## Solve using the JAX infrastructure

The density rises smoothly from approximately 1 to 4, reducing Alfvén speed
from approximately 1 to 1/2. A compact $\sin^8$ footpoint pulse is used instead
of $\sin^4$ to make its switch-on/off smoother ($C^7$), helping high-order spatial
convergence. Its support remains $0<t<T_d=0.5$ and amplitude is $10^{-3}L$.

`plan_alfven` forms $M=Q^TW\rho_0Q$ and $K=(B_0^2/\mu_0)G^TWG$ using
resolved BSPF trial functions at Gauss points. `integrate_alfven` evolves the
interior by RK4 with the full boundary mass lifting $-M_{ib}\ddot g$.
The default internal step is 0.0005. Explicit time stepping requires a stable
wave-CFL step; changing the grid or Alfvén speed can require smaller steps.


In [ ]:
def solve(n=129, rho=density, substeps=20, quadrature_order=8):
    z = jnp.linspace(0., 1., n)
    spatial = bspf_plans.plan_1d(z, degree=7, n_basis=24, boundary_points=9)
    model = bspf_alfven.plan_alfven(spatial, density=rho, quadrature_order=quadrature_order)
    xi, velocity = bspf_alfven.integrate_alfven(
        model, jnp.zeros_like(z), jnp.zeros_like(z), times,
        boundary=boundary, substeps=substeps)
    return z, spatial, model, xi, velocity

z, spatial, model, xi, velocity = solve()
magnetic = model.magnetic_field*bspf_operators.differentiate(spatial, xi.T).T
endpoint_error = float(jnp.max(jnp.abs(
    xi[:, jnp.array([0, -1])]-jax.vmap(boundary)(times))))
print(f"Displacement boundary residual: {endpoint_error:.3e}")
print(f"Maximum endpoint mismatch: {float(jnp.max(jnp.abs(xi[:, 0]-xi[:, -1]))):.3e}")
assert endpoint_error < 1e-14


## Independent uniform-density benchmark

Before interpreting variable-density scattering, check the same driven cavity
with $\rho_0=1$. The method of images gives an exact reference including repeated
wall reflections, with wave speed 1:
$$\xi_*(z,t)=\sum_{m\ge0}\{g(t-2m-z)-g(t-2m-2+z)\}.$$
The causal pulse makes all sufficiently delayed terms zero; four image pairs
are more than sufficient for the present $0\le t\le4$ window. This reference
uses no BSPF spatial operator or time integrator.


In [ ]:
_, _, _, uniform, _ = solve(rho=1.)
reference = sum(drive(times[:, None]-2*m-z[None, :])
                -drive(times[:, None]-2*m-2+z[None, :]) for m in range(4))
uniform_error = float(jnp.max(jnp.abs(uniform-reference))/amplitude)
print(f"Uniform-cavity error / drive amplitude: {uniform_error:.3e}")
assert uniform_error < 2e-5


## Spatial, time-step and quadrature refinement

The inhomogeneous problem has no imposed analytic solution. Compare complete
histories on shared grid points at 65, 129 and 257 samples, then halve the time
step and increase quadrature order independently. Errors are normalized by the
drive amplitude, so a small physical amplitude cannot hide poor accuracy.


In [ ]:
zc, _, _, coarse, _ = solve(n=65)
zf, _, _, fine, _ = solve(n=257)
_, _, _, half_step, _ = solve(substeps=40)
_, _, _, higher_quad, _ = solve(quadrature_order=10)
coarse_change = float(jnp.max(jnp.abs(coarse-xi[:, ::2]))/amplitude)
fine_change = float(jnp.max(jnp.abs(xi-fine[:, ::2]))/amplitude)
time_change = float(jnp.max(jnp.abs(xi-half_step))/amplitude)
quadrature_change = float(jnp.max(jnp.abs(xi-higher_quad))/amplitude)
print(f"65 -> 129 difference / amplitude: {coarse_change:.3e}")
print(f"129 -> 257 difference / amplitude: {fine_change:.3e}")
print(f"Halved-step change / amplitude: {time_change:.3e}")
print(f"Gauss 8 -> 10 change / amplitude: {quadrature_change:.3e}")
assert fine_change < 1e-5 and coarse_change > 20*fine_change
assert time_change < 1e-6 and quadrature_change < 1e-7


## Energy and boundary work using BSPF integration

$$E=\frac12\int_0^1\left(\rho_0 u_\perp^2+\frac{b_\perp^2}{\mu_0}\right)dz,
\qquad E'=[B_0u_\perp b_\perp/\mu_0]_0^1.$$
The right endpoint does no work. After $T_d$, both endpoints are stationary,
so the ideal system conserves energy even while waves reflect internally.

We use **`b.integrate` for the spatial energy integral** and
**`b.antiderivative` for accumulated boundary work**. The fields are evaluated on a denser integration grid before squaring, to
resolve the products. `interpolate(..., derivative=1)` differentiates the
original BSPF interpolant directly, without refitting derivative samples.
A second energy diagnostic
uses the same resolved quadrature as the weak solver, separating spatial
integration error from time-stepping drift. Boundary power is evaluated from
full BSPF endpoint derivatives, not inferred from the energy difference.


In [ ]:
# Evaluate the same BSPF fields on a denser grid BEFORE forming products.
energy_z = jnp.linspace(0., 1., 513)
energy_plan = bspf_plans.plan_1d(energy_z, degree=7, n_basis=24, boundary_points=9)
energy_velocity = bspf_calculus.interpolate(spatial, velocity.T, energy_z)
energy_magnetic = model.magnetic_field*bspf_calculus.interpolate(
    spatial, xi.T, energy_z, derivative=1)
energy_density = 0.5*(density(energy_z)[:, None]*energy_velocity**2
                     +energy_magnetic**2/model.permeability)
energy = bspf_calculus.integrate(energy_plan, energy_density)
weak_energy = bspf_alfven.alfven_energy(model, xi, velocity)
power = bspf_alfven.alfven_boundary_power(model, xi, velocity)
time_plan = bspf_plans.plan_1d(times, degree=7, n_basis=32, boundary_points=9)
work = bspf_calculus.antiderivative(time_plan, power)
scale = jnp.max(weak_energy)
after_drive = times >= drive_duration
balance_error = float(jnp.max(jnp.abs(energy-work))/scale)
integration_change = float(jnp.max(jnp.abs(energy-weak_energy))/scale)
energy_drift = float(jnp.max(jnp.abs(weak_energy[after_drive]
                                      -weak_energy[after_drive][0]))/scale)
print(f"Injected energy: {float(weak_energy[-1]):.9e}")
print(f"Relative boundary-work balance error: {balance_error:.3e}")
print(f"BSPF vs weak-quadrature energy: {integration_change:.3e}")
print(f"Relative weak-energy drift after drive: {energy_drift:.3e}")
assert balance_error < 2e-5 and integration_change < 2e-5
assert energy_drift < 1e-7


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].plot(z, density(z), label="Density")
axes[0, 0].plot(z, 1/jnp.sqrt(density(z)), label="Alfvén speed")
axes[0, 0].set(xlabel="z", title="Smooth density transition")
axes[0, 0].legend()
heat = axes[0, 1].pcolormesh(z, times, xi/amplitude, shading="auto", cmap="RdBu_r")
axes[0, 1].axhline(drive_duration, color="black", linestyle=":", linewidth=1)
axes[0, 1].set(xlabel="z", ylabel="t", title="Displacement / amplitude: scattering and reflection")
fig.colorbar(heat, ax=axes[0, 1])
for index in (25, 100, 200, 400):
    axes[1, 0].plot(z, magnetic[index], label=f"t={float(times[index]):g}")
axes[1, 0].set(xlabel="z", ylabel="Transverse magnetic perturbation", title="Magnetic field from BSPF differentiation")
axes[1, 0].legend()
axes[1, 1].plot(times, energy, label="BSPF spatial integral")
axes[1, 1].plot(times, work, "--", label="BSPF integrated boundary work")
axes[1, 1].axvline(drive_duration, color="gray", linestyle=":", label="Driver stops")
axes[1, 1].set(xlabel="t", ylabel="Energy", title="Injected energy remains in the cavity")
axes[1, 1].legend()
plt.show()
